In [27]:
import numpy as np
import tensorflow as tf
from tensorflow import keras

import keras
from keras import layers
from keras import ops
from tensorflow.keras.models import Sequential
import keras_tuner as kt

from tensorflow.keras import layers, Model
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

print(f"TensorFlow {tf.__version__}")
print(f"GPUs: {tf.config.list_physical_devices('GPU')}")

TensorFlow 2.20.0
GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [28]:
# load data 
d = np.load("processed_waveforms.npz")
 # ["X_euclidean"] or ["X_voltage"] 
X = d["X_euclidean"].astype(np.float32)  
# 0=photon, 1=neutron
y = d["y"].astype(np.int32)             

# split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

#Normalize to [0, 1]
minv = X_train.min(axis=0, keepdims=True)    
maxv = X_train.max(axis=0, keepdims=True)   
den  = np.maximum(maxv - minv, 1e-8)

X_train_n = (X_train - minv) / den
X_test_n  = (X_test  - minv) / den

In [ ]:
#Initial Basic Autoencoder 
class DenseAutoencoder(Model):
    def __init__(self, n_features=104, latent_dim=16):
        super().__init__()
        
        self.encoder = tf.keras.Sequential([
            layers.Input(shape=(n_features,)),
            layers.Dense(64, activation="relu"),
            layers.Dense(32, activation="relu"),
            layers.Dense(16, activation="relu"),
            layers.Dense(8, activation="relu"),
            layers.Dense(latent_dim, activation="relu"),
        ])
        self.decoder = tf.keras.Sequential([
            layers.Input(shape=(latent_dim,)),
            layers.Dense(16, activation="relu"),
            layers.Dense(32, activation="relu"),
            layers.Dense(64, activation="relu"),
            layers.Dense(n_features, activation="sigmoid"),  # output in [0,1]
        ])

    def call(self, x):
        return self.decoder(self.encoder(x))

#initialize and compile autoencoder 
ae = DenseAutoencoder(n_features=104, latent_dim=16)
ae.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss="mae")


history = ae.fit(
    X_train_n, X_train_n,
    validation_data=(X_test_n, X_test_n),
    epochs=30,
    batch_size=512,
    shuffle=True,
    callbacks=[tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True)],
    verbose=1
)

# Loss plot 
plt.figure()
plt.plot(history.history["loss"], label="train")
plt.plot(history.history["val_loss"], label="val")
plt.legend()
plt.savefig("figures/a1_initial_training_loss.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
import os; os.makedirs("figures", exist_ok=True)
# Reconstruct and plot several random waveforms.
# Different pulses are picked each run so the plots are fresh.
n_plot = 6
rng_plot = np.random.default_rng()  # no seed — random each run
indices = rng_plot.choice(X_test_n.shape[0], size=n_plot, replace=False)

# Predict in min-max [0,1] space, invert the min-max to get back to L2-normed scale
recon_n = ae.predict(X_test_n[indices], verbose=0)
recon_l2 = recon_n * den[0] + minv[0]       # back to L2-normed space
orig_l2  = X_test[indices]                   # already in L2-normed space

# Try to convert both to raw voltage using the stored per-waveform L2 norms.
# Requires preprocess.py to have been re-run to save X_norms.
try:
    norms_all = np.load("processed_waveforms.npz")["X_norms"]
    _, norms_test = train_test_split(norms_all, test_size=0.2, random_state=42, stratify=y)
    orig_raw  = orig_l2  * norms_test[indices, None]
    recon_raw = recon_l2 * norms_test[indices, None]
    show_raw = True
except KeyError:
    show_raw = False

fig, axes = plt.subplots(2, 3, figsize=(14, 7), sharex=True)
axes = axes.flatten()
for ax, row, idx in zip(axes, range(n_plot), indices):
    if show_raw:
        ax.plot(orig_raw[row],  label="Original",      lw=1.8, color="tab:blue")
        ax.plot(recon_raw[row], label="Reconstructed", lw=1.8, linestyle="--", color="tab:orange")
        ax.set_ylabel("Voltage [V]")
    else:
        ax.plot(orig_l2[row],  label="Original",      lw=1.8, color="tab:blue")
        ax.plot(recon_l2[row], label="Reconstructed", lw=1.8, linestyle="--", color="tab:orange")
        ax.set_ylabel("L2-normalized")
    label = "neutron" if y_test[idx] == 1 else "photon"
    ax.set_title(f"test idx {idx} ({label})", fontsize=10)
    ax.set_xlabel("Sample")
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=8, loc="upper right")

scale_str = "raw voltage" if show_raw else "L2-normalized space"
fig.suptitle(f"Dense autoencoder reconstructions ({scale_str})", fontsize=13)
plt.tight_layout()
plt.savefig("figures/a1_initial_reconstructions.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
#For demonstration I am using the Logistic regression classifier
#we could investigate better methods

#  Extract low-dim feature vectors from autoencoder
Z_train = ae.encoder(X_train_n).numpy()  
Z_test  = ae.encoder(X_test_n).numpy() 

#  Logistic regression classifier  
clf = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=5000, class_weight="balanced")
)

#Fit Logistic regression classifier 
clf.fit(Z_train, y_train)

#Predictions 
y_pred = clf.predict(Z_test)
p_neutron = clf.predict_proba(Z_test)[:, 1]


cm = confusion_matrix(y_test, y_pred, labels=[0, 1])

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["photon (0)", "neutron (1)"]
)
disp.plot(values_format="d", cmap="Blues")
plt.title("Confusion Matrix")
plt.savefig("figures/a1_initial_confusion_matrix.png", dpi=120, bbox_inches="tight")
plt.show()


tn, fp, fn, tp = cm.ravel()

 # true photon predicted neutron
photon_misclass_rate  = ( fp / (tn + fp) ) * 100
# true neutron predicted photon
neutron_misclass_rate = (fn / (tp + fn) ) * 100 

print(f"Photon misclassification rate (photon→neutron): {photon_misclass_rate:.4f}")
print(f"Neutron misclassification rate (neutron→photon): {neutron_misclass_rate:.4f}")